# Car Price Prediction — Model Training

In this notebook, we will train and evaluate machine learning models to predict vehicle prices.

We will start by splitting the cleaned dataset into training and testing sets, encoding categorical features, and establishing a baseline.

We will then train regression models and compare their predictive performance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor

import joblib

## Loading the Cleaned Dataset

We load the processed dataset generated during data preprocessing.

This dataset contains the nine selected features and the target variable (`price`), with missing values and identified inconsistencies already handled.

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

file_path = project_root / "data" / "processed" / "cars_clean.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (108273, 10)


,mileage_numeric,vehicle_age,power_hp,make,model,fuel_category,transmission,body_type,country_code,price
0,10500.0,1.0,83.0,Suzuki,Swift,Gasoline,Manual,Compact,NL,22950.0
1,55609.0,5.0,360.0,BMW,X3 M,Gasoline,Automatic,Off-Road/Pick-up,DE,46000.0
2,18200.0,7.0,500.0,Porsche,991,Gasoline,Manual,Coupe,BE,179900.0
3,10.0,0.0,204.0,Audi,A6,Electric/Diesel,Automatic,Station wagon,IT,70400.0
4,74602.0,6.0,330.0,Porsche,Panamera,Electric/Gasoline,Automatic,Station wagon,NL,66999.0


## Preparing Features and Target

We separate the selected vehicle characteristics (X) from the target variable (y), which represents the vehicle price.

The model will use nine features to learn how different vehicle characteristics influence the advertised price.

In [3]:
numerical_features = ["mileage_numeric", "vehicle_age", "power_hp"]

categorical_features = ["make", "model", "fuel_category", "transmission", "body_type", "country_code"]

features = numerical_features + categorical_features

X = df[features].copy()
y = df["price"].copy()

print("Features:", X.shape)
print("Target:", y.shape)

Features: (108273, 9)
Target: (108273,)


## Train-Test Split

We split the dataset into 80% training data and 20% testing data.

The training set will be used to train our models, while the testing set will be used to evaluate predictions on unseen vehicles.

A fixed random state ensures reproducible results.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 86618
Testing records: 21655


## Categorical Encoding

We use One-Hot Encoding to convert categorical features into numerical representations without introducing an artificial order between categories.

Numerical features remain unchanged. The encoder is fitted only on the training data to prevent data leakage.

Other encoding techniques may be explored during future model optimization.

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

Training shape: (86618, 662)
Testing shape: (21655, 662)


## Baseline Model

Before training machine learning algorithms, we establish a baseline using the median vehicle price from the training set.

The baseline predicts the same price for every vehicle, regardless of its characteristics.

We will evaluate its performance using Mean Absolute Error (MAE), which measures the average absolute difference between actual and predicted prices.

This result will serve as a reference for comparing more advanced regression models.

In [6]:
# Create and train the baseline model
baseline_model = DummyRegressor(strategy="median")
baseline_model.fit(X_train_encoded, y_train)

# Predict car prices
baseline_predictions = baseline_model.predict(X_test_encoded)

# Calculate the prediction error
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Training median price: €{y_train.median():,.2f}")
print(f"Baseline MAE: €{baseline_mae:,.2f}")

# Compare actual and predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": baseline_predictions
})

comparison.head()

Training median price: €39,485.00
Baseline MAE: €34,978.86


,Actual Price,Predicted Price
0,44890.0,39485.0
1,1850.0,39485.0
2,97990.0,39485.0
3,150900.0,39485.0
4,1199.0,39485.0


### Baseline Results

The baseline predicts a constant price of €39,485 for every vehicle.

It achieved a Mean Absolute Error (MAE) of €34,978.86 on the test set.

This relatively high error reflects the limitations of predicting a single value for vehicles with widely varying prices.

We will use this result as a reference to evaluate whether more advanced regression models can produce better predictions.

## Linear Regression

We will train a Linear Regression model to predict vehicle prices using the selected numerical and categorical features.

Unlike the baseline, Linear Regression learns relationships between vehicle characteristics and their prices.

We will evaluate its performance using Mean Absolute Error (MAE) and compare the result with our baseline.

In [7]:
# Create and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)

# Predict car prices
linear_predictions = linear_model.predict(X_test_encoded)

# Calculate the prediction error
linear_mae = mean_absolute_error(y_test, linear_predictions)

print(f"Baseline MAE: €{baseline_mae:,.2f}")
print(f"Linear Regression MAE: €{linear_mae:,.2f}")

# Compare actual and predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": linear_predictions
})

comparison.head()

Baseline MAE: €34,978.86
Linear Regression MAE: €20,882.85


,Actual Price,Predicted Price
0,44890.0,47210.684777
1,1850.0,-3175.488160
2,97990.0,76057.288013
3,150900.0,142783.477976
4,1199.0,-9783.402354


### Linear Regression Results

The Linear Regression model achieved a MAE of €20,882.85, reducing the baseline error by approximately 40.3%.

Although the model significantly improved prediction accuracy, some predictions were negative, which is not realistic for vehicle prices.

We will investigate how frequently negative predictions occur before deciding whether additional improvements are necessary.

In [8]:
negative_predictions = linear_predictions < 0

print("Total predictions:", len(linear_predictions))
print("Negative predictions:", negative_predictions.sum())
print("Negative predictions (%):", round(negative_predictions.mean() * 100, 2))

Total predictions: 21655
Negative predictions: 1690
Negative predictions (%): 7.8


**Negative Predictions — Findings**

Of the 21,655 test predictions, 1,690 (7.8%) were negative.

Although Linear Regression reduced the baseline MAE by approximately 40.3%, these negative predictions reveal an important limitation: the model does not guarantee realistic, positive vehicle prices.

We will keep these results unchanged and compare them with a more flexible regression algorithm.

## Random Forest Regression

We will train a Random Forest Regressor to capture more complex relationships between vehicle characteristics and prices.

Unlike Linear Regression, Random Forest combines predictions from multiple decision trees and can model non-linear patterns.

We will evaluate its performance using MAE and compare it with the Baseline and Linear Regression models.

In [9]:
# Create and train the Random Forest model
forest_model = RandomForestRegressor(n_estimators=50, max_depth=18, min_samples_leaf=3, random_state=42, n_jobs=2)
forest_model.fit(X_train_encoded, y_train)

# Predict car prices
forest_predictions = forest_model.predict(X_test_encoded)

# Calculate the prediction error
forest_mae = mean_absolute_error(y_test, forest_predictions)

print(f"Baseline MAE: €{baseline_mae:,.2f}")
print(f"Linear Regression MAE: €{linear_mae:,.2f}")
print(f"Random Forest MAE: €{forest_mae:,.2f}")

# Compare actual and predicted prices
forest_comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": forest_predictions
})

forest_comparison.head()

Baseline MAE: €34,978.86
Linear Regression MAE: €20,882.85
Random Forest MAE: €6,359.84


,Actual Price,Predicted Price
0,44890.0,41094.916011
1,1850.0,3724.120633
2,97990.0,92543.789256
3,150900.0,148492.215239
4,1199.0,2500.669122


### Random Forest Results

The Random Forest model achieved a MAE of €6,359.84, reducing the error by approximately 69.5% compared to Linear Regression and 81.8% compared to the Baseline.

The model produced more accurate predictions and avoided the negative price predictions observed with Linear Regression.

These results suggest that non-linear relationships are important for vehicle price prediction.

However, we still need to check whether the model is overfitting the training data.

### Checking for Overfitting

We will compare the Random Forest's performance on the training and testing datasets.

A significantly lower training error may indicate that the model has learned patterns specific to the training data rather than generalizing well to unseen vehicles.

This comparison will help us assess whether the model needs further adjustments.

In [10]:
# Predict prices for training vehicles
forest_train_predictions = forest_model.predict(X_train_encoded)

# Calculate training error
forest_train_mae = mean_absolute_error(y_train, forest_train_predictions)

# Compare training and testing errors
print(f"Random Forest Training MAE: €{forest_train_mae:,.2f}")
print(f"Random Forest Testing MAE: €{forest_mae:,.2f}")

Random Forest Training MAE: €4,915.56
Random Forest Testing MAE: €6,359.84


### Overfitting Analysis — Findings

The Random Forest achieved a Training MAE of €4,915.56 and a Testing MAE of €6,359.84.

The testing error is approximately 29.4% higher than the training error, indicating a moderate performance gap.

Although the model performs better on the training data, the results do not provide clear evidence of severe overfitting.

Further validation would help assess its ability to generalize to unseen vehicles.

## Train-Validation Split

We split the existing training data into smaller training (80%) and validation (20%) subsets.

The training subset will be used to train new models, while the validation subset will help compare their performance and select suitable configurations.

The original test set will remain separate from these experiments.

In [6]:
X_train_small, X_val, y_train_small, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print("Training records:", len(X_train_small))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

Training records: 69294
Validation records: 17324
Testing records: 21655


## CatBoost Regression

We will train a CatBoost Regressor using our nine original features, without applying manual One-Hot Encoding.

CatBoost handles categorical features internally and uses gradient boosting to learn complex relationships between vehicle characteristics and prices.

We will evaluate the model on the validation set using Mean Absolute Error (MAE).

In [12]:
# Create the model
catboost_model = CatBoostRegressor(cat_features=categorical_features, verbose=False)

# Train the model
catboost_model.fit(X_train_small, y_train_small)

# Make predictions
catboost_predictions = catboost_model.predict(X_val)

# Calculate the error
catboost_mae = mean_absolute_error(y_val, catboost_predictions)

print(f"CatBoost Validation MAE: €{catboost_mae:,.2f}")

CatBoost Validation MAE: €6,965.46


### Comparing Default Models

We will first compare Random Forest and CatBoost using their default hyperparameters.

Both models will use the same training and validation datasets, and their performance will be measured using MAE.

After establishing this initial comparison, we will experiment with hyperparameter tuning for both algorithms to determine whether their performance can be improved.

In [13]:
# Encode the new training and validation datasets
validation_preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features)
])

X_train_small_encoded = validation_preprocessor.fit_transform(X_train_small)
X_val_encoded = validation_preprocessor.transform(X_val)

In [14]:
# Create the Random Forest with default parameters
forest_default = RandomForestRegressor(random_state=42, n_jobs=2)

# Train the model
forest_default.fit(X_train_small_encoded, y_train_small)

# Make predictions
forest_default_predictions = forest_default.predict(X_val_encoded)

# Calculate the error
forest_default_mae = mean_absolute_error(y_val, forest_default_predictions)

# Compare default models
print(f"Random Forest Default MAE: €{forest_default_mae:,.2f}")
print(f"CatBoost Default MAE: €{catboost_mae:,.2f}")

Random Forest Default MAE: €5,844.60
CatBoost Default MAE: €6,965.46


### Default Random Forest — Findings

The default Random Forest achieved a validation MAE of €5,844.60, improving on the previously configured Random Forest (€6,517.48).

This shows that the manually selected hyperparameters did not improve performance in our initial experiments.

Random Forest also outperformed the initial CatBoost configuration (€6,965.46), although further tuning is needed before selecting the final model.

### CatBoost — Number of Iterations

Our initial CatBoost model used 500 iterations and achieved a validation MAE of €6,965.46.

We will increase the number of iterations to 1,000 while keeping the remaining model settings unchanged.

This experiment will help us determine whether additional boosting iterations improve prediction accuracy.

In [15]:
# Create CatBoost with more iterations
catboost_tuned = CatBoostRegressor(
    iterations=1000,
    loss_function="MAE",
    cat_features=categorical_features,
    random_seed=42,
    verbose=False
)

# Train the model
catboost_tuned.fit(X_train_small, y_train_small)

# Make predictions
catboost_tuned_predictions = catboost_tuned.predict(X_val)

# Calculate the error
catboost_tuned_mae = mean_absolute_error(y_val, catboost_tuned_predictions)

# Compare the results
print(f"CatBoost (1000 iterations): €{catboost_tuned_mae:,.2f}")


CatBoost (1000 iterations): €7,703.76


### CatBoost — Training Objective Experiment

The initial CatBoost model achieved a validation MAE of €6,965.46 using its default RMSE training objective.

A second configuration using MAE as the training objective and a fixed random seed achieved a validation MAE of €7,703.76.

The second configuration performed worse. However, since both the training objective and random seed changed, the difference cannot be attributed to a single parameter.

The default Random Forest remains the best-performing model so far, with a validation MAE of €5,844.60.

### CatBoost — Tree Depth Experiment

We will investigate whether increasing the tree depth from 6 to 8 improves the CatBoost model.

All other settings will remain the same as in the initial configuration, allowing us to evaluate the effect of tree depth on validation performance.

In [16]:
# Create the model with deeper trees
catboost_depth8 = CatBoostRegressor(cat_features=categorical_features, depth=8, verbose=False)

# Train the model
catboost_depth8.fit(X_train_small, y_train_small)

# Make predictions
depth8_predictions = catboost_depth8.predict(X_val)

# Calculate the error
depth8_mae = mean_absolute_error(y_val, depth8_predictions)

print(f"CatBoost (depth 8) MAE: €{depth8_mae:,.2f}")

CatBoost (depth 8) MAE: €6,176.70


### Tree Depth Experiment — Findings

Increasing CatBoost's tree depth from 6 to 8 reduced the validation MAE from €6,965.46 to €6,176.70, an improvement of approximately 11.3%.

This suggests that deeper trees helped the model capture more complex relationships between vehicle characteristics and prices.

However, the default Random Forest still achieved a lower validation MAE of €5,844.60.

CatBoost with depth 8 is currently our best CatBoost configuration, although further validation is needed before selecting the final model.

### Checking Overfitting in the Best Models

We will compare the training and validation errors of our two strongest models: the default Random Forest and CatBoost with depth 8.

This will help us identify whether either model is fitting the training data too closely.

We will reuse the existing trained models without repeating the training process.

In [17]:
# Predict training prices using the existing models
forest_train_predictions = forest_default.predict(X_train_small_encoded)
catboost_train_predictions = catboost_depth8.predict(X_train_small)

# Calculate training errors
forest_train_mae = mean_absolute_error(y_train_small, forest_train_predictions)
catboost_train_mae = mean_absolute_error(y_train_small, catboost_train_predictions)

print(f"Random Forest Training MAE: €{forest_train_mae:,.2f}")
print(f"CatBoost Depth 8 Training MAE: €{catboost_train_mae:,.2f}")

Random Forest Training MAE: €2,237.16
CatBoost Depth 8 Training MAE: €5,270.38


### Overfitting Analysis — Findings

The default Random Forest achieved a training MAE of €2,237.16 and a validation MAE of €5,844.60, showing a substantial performance gap.

CatBoost with depth 8 achieved a training MAE of €5,270.38 and a validation MAE of €6,176.70, showing a smaller gap between training and validation performance.

These results suggest that Random Forest is more prone to overfitting, while CatBoost generalizes more consistently across the two datasets.

However, Random Forest still achieves the lowest validation MAE, so both models remain relevant for further evaluation.

## Error Analysis by Price Range

Overall MAE does not show whether a model performs equally well across different vehicle price ranges.

We will compare Random Forest and CatBoost prediction errors for lower-priced, mid-range, and luxury vehicles.

This analysis will help us understand where each model performs well and where prediction accuracy could be improved.

In [18]:
# Create a table with actual prices and prediction errors
error_analysis = pd.DataFrame({
    "Actual Price": y_val.to_numpy(),
    "Random Forest Error": abs(y_val.to_numpy() - forest_default_predictions),
    "CatBoost Error": abs(y_val.to_numpy() - depth8_predictions)
})

# Group vehicles by price range
error_analysis["Price Range"] = pd.cut(
    error_analysis["Actual Price"],
    bins=[0, 10000, 30000, 60000, 100000, float("inf")],
    labels=["Under €10k", "€10k-30k", "€30k-60k", "€60k-100k", "Over €100k"],
    include_lowest=True
)

# Compare the average error in each price range
error_analysis.groupby("Price Range", observed=True).agg(
    Cars=("Actual Price", "count"),
    Random_Forest_MAE=("Random Forest Error", "mean"),
    CatBoost_MAE=("CatBoost Error", "mean")
).round(2)

,Cars,Random_Forest_MAE,CatBoost_MAE
Price Range,,,
Under €10k,1322,1404.25,1672.93
€10k-30k,5101,2280.96,2416.99
€30k-60k,5404,3734.22,4142.94
€60k-100k,3357,6065.19,6765.48
Over €100k,2140,22065.28,22132.85


### Error Analysis by Price Range — Findings

Random Forest achieved a lower MAE in four of the five price ranges, outperforming CatBoost for vehicles priced up to €100,000.

CatBoost performed slightly better for vehicles above €100,000, with a MAE of €21,332.85 compared to €22,065.28 for Random Forest.

Both models showed increasing absolute errors in higher price ranges.

However, absolute errors alone do not show how significant these errors are relative to vehicle prices. We will therefore examine percentage errors to better understand model performance.

## Saving Model Checkpoints

Training machine learning models can be computationally expensive.

To avoid repeating completed training sessions, we save our current trained models, preprocessing encoder, and validation results.

These checkpoints allow us to resume model evaluation in future sessions without retraining the same models.

In [19]:
# Create a folder for saved models
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

# Save the current models and results
checkpoint = {
    "forest_default": forest_default,
    "catboost_depth8": catboost_depth8,
    "validation_preprocessor": validation_preprocessor,
    "y_val": y_val,
    "forest_default_predictions": forest_default_predictions,
    "depth8_predictions": depth8_predictions,
    "error_analysis": error_analysis
}

joblib.dump(checkpoint, models_dir / "checkpoint.joblib")

print("Checkpoint saved successfully!")

Checkpoint saved successfully!


### Percentage Error Analysis

Absolute prediction errors tend to increase for more expensive vehicles, making comparisons between price ranges difficult.

We will calculate the absolute percentage error for each prediction and compare the median percentage error across price ranges.

This will help us understand how prediction accuracy changes relative to the actual vehicle price.

In [20]:
# Calculate percentage errors
error_analysis["Random Forest Error (%)"] = error_analysis["Random Forest Error"] / error_analysis["Actual Price"] * 100
error_analysis["CatBoost Error (%)"] = error_analysis["CatBoost Error"] / error_analysis["Actual Price"] * 100

# Compare median percentage errors by price range
error_analysis.groupby("Price Range", observed=True)[["Random Forest Error (%)", "CatBoost Error (%)"]].median().round(2)

,Random Forest Error (%),CatBoost Error (%)
Price Range,,
Under €10k,18.87,23.93
€10k-30k,7.82,8.74
€30k-60k,6.43,7.61
€60k-100k,5.60,6.50
Over €100k,6.85,7.05


### Percentage Error Analysis — Findings

Random Forest achieved lower median absolute percentage errors than CatBoost across all five price ranges.

Both models showed their highest relative errors for vehicles priced below €10,000, while their lowest errors occurred in the €60,000–€100,000 range.

Although CatBoost achieved a lower absolute MAE for vehicles above €100,000, Random Forest still had a slightly lower median percentage error in this range.

These findings reinforce Random Forest as our strongest model so far, while highlighting that lower-priced vehicles remain more challenging to predict accurately.

### CatBoost — Iterations and Learning Rate

We will test a CatBoost configuration with 1,500 iterations and a learning rate of 0.05.

A lower learning rate allows the model to make smaller adjustments during training, while additional iterations provide more opportunities to improve predictions.

We will keep the tree depth at 8 and evaluate whether this combination improves validation MAE.

In [7]:
# Create the model
catboost_b = CatBoostRegressor(
    iterations=1500,
    depth=8,
    learning_rate=0.05,
    cat_features=categorical_features,
    verbose=False,
    allow_writing_files=False
)

# Train the model
catboost_b.fit(X_train_small, y_train_small)

# Make predictions
predictions_b = catboost_b.predict(X_val)

# Calculate the error
mae_b = mean_absolute_error(y_val, predictions_b)

print(f"CatBoost B Validation MAE: €{mae_b:,.2f}")

CatBoost B Validation MAE: €6,160.89


### CatBoost — Increasing Tree Depth

We will increase the tree depth from 8 to 10 while keeping the iterations and learning rate unchanged.

This experiment will help us determine whether more complex trees improve prediction accuracy.

We will compare both configurations using the same validation dataset and MAE metric.

In [8]:
# Create the model
catboost_c = CatBoostRegressor(
    iterations=1500,
    depth=10,
    learning_rate=0.05,
    cat_features=categorical_features,
    verbose=False,
    allow_writing_files=False
)

# Train the model
catboost_c.fit(X_train_small, y_train_small)

# Make predictions
predictions_c = catboost_c.predict(X_val)

# Calculate the error
mae_c = mean_absolute_error(y_val, predictions_c)

print(f"CatBoost C Validation MAE: €{mae_c:,.2f}")

CatBoost C Validation MAE: €5,823.87


### Final CatBoost Experiments — Findings

CatBoost B achieved a validation MAE of €6,160.89, slightly improving on the previous depth 8 configuration (€6,176.70).

Increasing the tree depth from 8 to 10 in CatBoost C further reduced the validation MAE to €5,823.87.

CatBoost C achieved the lowest validation MAE so far, slightly outperforming the default Random Forest (€5,844.60).

However, the difference is only €20.73 (approximately 0.35%), so further evaluation is needed before selecting the final model.

### Checking CatBoost C for Overfitting

CatBoost C achieved the lowest validation MAE in our experiments.

Before selecting the final model, we will compare its training and validation errors to investigate potential overfitting.

This check reuses the trained model without requiring additional training.

In [9]:
# Predict prices for training vehicles
catboost_c_train_predictions = catboost_c.predict(X_train_small)

# Calculate training error
catboost_c_train_mae = mean_absolute_error(y_train_small, catboost_c_train_predictions)

print(f"CatBoost C Training MAE: €{catboost_c_train_mae:,.2f}")
print(f"CatBoost C Validation MAE: €{mae_c:,.2f}")

CatBoost C Training MAE: €4,771.53
CatBoost C Validation MAE: €5,823.87


In [10]:
joblib.dump(catboost_c, project_root / "models" / "catboost_c.joblib")

['C:\\Users\\fvfra\\PycharmProjects\\car-price-prediction\\models\\catboost_c.joblib']

### CatBoost C — Overfitting Analysis Findings

CatBoost C achieved a training MAE of €4,771.53 and a validation MAE of €5,823.87, resulting in a generalization gap of €1,052.34.

In comparison, the default Random Forest showed a much larger gap of €3,607.44 between training and validation MAE.

These results suggest that CatBoost C is less affected by overfitting, while also achieving a slightly lower validation MAE.

Although the validation performance difference is small, CatBoost C is our preferred candidate for the final model. Its performance on unseen test data still needs to be evaluated.

## CatBoost Optimization — Early Stopping

CatBoost C achieved the best validation MAE so far (€5,823.87) and was selected as our preferred model.

Before finalizing the model, we will explore whether additional boosting iterations can improve its performance.

We will introduce **early stopping**, which monitors validation performance and stops training when no further improvement is observed.

This experiment keeps the tree depth and learning rate unchanged while allowing up to 2,500 iterations.

In [7]:
catboost_d = CatBoostRegressor(
    iterations=2500,
    depth=10,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="MAE",
    cat_features=categorical_features,
    verbose=False,
    allow_writing_files=False
)

# Train with early stopping
catboost_d.fit(
    X_train_small,
    y_train_small,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    use_best_model=True
)

# Evaluate the model
predictions_d = catboost_d.predict(X_val)
mae_d = mean_absolute_error(y_val, predictions_d)

print(f"CatBoost D Validation MAE: €{mae_d:,.2f}")
print(f"Best iteration: {catboost_d.best_iteration_}")

CatBoost D Validation MAE: €5,614.85
Best iteration: 2499


### CatBoost D — Findings

CatBoost D achieved a validation MAE of €5,614.85, improving on CatBoost C by €209.02 (approximately 3.59%).

The best iteration was 2499, corresponding to the last of the 2,500 allowed iterations. Therefore, early stopping was not triggered.

This suggests that additional iterations may further improve validation performance.

We will continue training from the existing model to investigate this possibility without repeating the initial 2,500 iterations.

### CatBoost E — Continuing Training

Since CatBoost D achieved its best validation MAE at the final allowed iteration, we will investigate whether additional trees improve performance.

Instead of restarting training, we will use the previously trained CatBoost D model as the starting point and allow up to 1,500 additional iterations.

Early stopping will remain enabled to reduce unnecessary training when validation performance stops improving.

In [10]:
catboost_e = CatBoostRegressor(
    iterations=1500,
    depth=10,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="MAE",
    cat_features=categorical_features,
    verbose=False,
    allow_writing_files=False
)

# Continue training from CatBoost D
catboost_e.fit(
    X_train_small,
    y_train_small,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    use_best_model=True,
    init_model=catboost_d
)

# Evaluate the extended model
predictions_e = catboost_e.predict(X_val)
mae_e = mean_absolute_error(y_val, predictions_e)

print(f"CatBoost E Validation MAE: €{mae_e:,.2f}")
print(f"Total trees: {catboost_e.tree_count_}")

# Save the new model
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(catboost_e, models_dir / "catboost_e.joblib")

CatBoost E Validation MAE: €5,562.18
Total trees: 3987


['C:\\Users\\fvfra\\PycharmProjects\\car-price-prediction\\models\\catboost_e.joblib']

In [11]:
print(f"Best iteration: {catboost_e.best_iteration_}")
print(f"Total trees: {catboost_e.tree_count_}")

Best iteration: 1486
Total trees: 3987


### CatBoost F — Further Training with Early Stopping

CatBoost E achieved a validation MAE of €5,562.18 using 3,987 trees.

Its best iteration occurred near the maximum allowed number of additional iterations, suggesting that further training might still improve performance.

We will extend training once more, using early stopping to monitor whether the validation MAE stabilizes.

In [12]:
catboost_f = CatBoostRegressor(
    iterations=1500,
    depth=10,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="MAE",
    cat_features=categorical_features,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Continue from CatBoost E
catboost_f.fit(
    X_train_small,
    y_train_small,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    use_best_model=True,
    init_model=catboost_e
)

# Evaluate
predictions_f = catboost_f.predict(X_val)
mae_f = mean_absolute_error(y_val, predictions_f)

print(f"CatBoost F Validation MAE: €{mae_f:,.2f}")
print(f"Best iteration: {catboost_f.best_iteration_}")
print(f"Total trees: {catboost_f.tree_count_}")

# Save the model
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(catboost_f, models_dir / "catboost_f.joblib")

print("CatBoost F saved successfully!")

CatBoost F Validation MAE: €5,562.53
Best iteration: 0
Total trees: 3988
CatBoost F saved successfully!


### CatBoost F — Findings and Model Selection

CatBoost F achieved a validation MAE of €5,562.53 using 3,988 trees.

The best iteration occurred at the beginning of the additional training phase, and early stopping prevented further unnecessary training.

Since CatBoost F did not improve upon the saved CatBoost E (€5,562.18), we selected **CatBoost E as our preferred configuration**.

Further increasing the number of iterations is unlikely to provide meaningful improvements under the current settings.

We will now retrain the selected configuration using the complete training dataset and evaluate its performance on the test dataset.

## Final Model Training

After completing the hyperparameter experiments, CatBoost was selected as our final regression algorithm.

The selected configuration uses a tree depth of 10, a learning rate of 0.05, and approximately 4,000 boosting iterations.

We will retrain the model using the entire training dataset, combining the previously separated training and validation subsets.

The test dataset will remain excluded from training and will be used for the final performance evaluation.

In [13]:
# Create the final CatBoost model
final_model = CatBoostRegressor(
    iterations=3987,
    depth=10,
    learning_rate=0.05,
    loss_function="RMSE",
    cat_features=categorical_features,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Train on the complete training dataset
final_model.fit(X_train, y_train)

# Save the final model
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, models_dir / "catboost_final.joblib")

print("Final CatBoost model trained and saved successfully!")

Final CatBoost model trained and saved successfully!


In [5]:
final_model = joblib.load(
    project_root / "models" / "catboost_final.joblib"
)

print("Final model loaded successfully!")

Final model loaded successfully!


## Final Model Evaluation

After completing hyperparameter optimization, the selected CatBoost configuration was retrained on the full training dataset of 86,618 vehicles.

The final model uses 3,987 iterations, a tree depth of 10, and a learning rate of 0.05.

We will now evaluate its performance on the 21,655 vehicles in the test dataset using three metrics:

- **MAE (Mean Absolute Error):** The average difference between predicted and actual prices, in euros. Lower is better.
- **RMSE (Root Mean Squared Error):** Similar to MAE, but penalizes larger prediction errors more heavily. Lower is better.
- **R² (R-squared):** Measures how well the model explains variations in vehicle prices. Values closer to 1 indicate better performance.

Although the test dataset was not used to train the final model, it was inspected during earlier experiments. Therefore, this evaluation is not based on a completely untouched test set.

In [7]:
# Predict prices on the test dataset
final_predictions = final_model.predict(X_test)

# Calculate evaluation metrics
final_mae = mean_absolute_error(y_test, final_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
final_r2 = r2_score(y_test, final_predictions)

# Display results
print(f"Final Test MAE: €{final_mae:,.2f}")
print(f"Final Test RMSE: €{final_rmse:,.2f}")
print(f"Final Test R²: {final_r2:.4f}")

Final Test MAE: €5,686.50
Final Test RMSE: €30,342.40
Final Test R²: 0.9218


### Final Model Evaluation — Findings

The final CatBoost model achieved the following results on the test dataset:

- **MAE:** €5,686.50
- **RMSE:** €30,342.40
- **R²:** 0.9218

The model explains approximately 92.18% of the variation in vehicle prices, indicating strong overall predictive performance.

The test MAE is close to the validation MAE (€5,562.18), suggesting reasonably consistent performance on unseen data.

However, the RMSE is substantially higher than the MAE, indicating that some predictions may have very large errors.

Overall, the results are promising, although the model's limitations and unusually large prediction errors deserve further investigation.

## Analyzing the Largest Prediction Errors

The final model achieved a significantly higher RMSE than MAE, suggesting that some predictions have unusually large errors.

To investigate this, we will identify the five vehicles with the largest absolute prediction errors.

By comparing their actual prices, predicted prices, and vehicle characteristics, we can better understand the model's limitations.

In [8]:
# Create a DataFrame with the test vehicles
error_cases = X_test.copy().reset_index(drop=True)

# Add actual and predicted prices
error_cases["Actual Price"] = y_test.to_numpy()
error_cases["Predicted Price"] = final_predictions

# Calculate absolute errors
error_cases["Absolute Error"] = (
    error_cases["Actual Price"] - error_cases["Predicted Price"]
).abs()

# Display the five largest prediction errors
error_cases.sort_values(
    "Absolute Error", ascending=False
)[[
    "make", "model", "vehicle_age", "mileage_numeric",
    "Actual Price", "Predicted Price", "Absolute Error"
]].head(5).round(2)

,make,model,vehicle_age,mileage_numeric,Actual Price,Predicted Price,Absolute Error
9995,Bugatti,Chiron,2.0,1628.0,6800000.0,4905241.21,1894758.79
17996,Bugatti,Chiron,1.0,2111.0,5400000.0,3793691.55,1606308.45
3020,Mercedes-Benz,AMG ONE,0.0,10.0,3629950.0,2049576.04,1580373.96
18057,Bugatti,Chiron,4.0,5293.0,5250000.0,3895616.57,1354383.43
9308,Aston Martin,Valkyrie,1.0,80.0,4300000.0,2957992.15,1342007.85


### Largest Prediction Errors — Findings

The five largest prediction errors occurred in extremely expensive vehicles, including the Bugatti Chiron, Mercedes-Benz AMG ONE, and Aston Martin Valkyrie.

All five vehicles had actual prices above €3.6 million, and the model underestimated their prices by approximately €1.3 million to €1.9 million.

These extreme errors help explain why the RMSE (€30,342.40) is significantly higher than the MAE (€5,686.50).

A possible explanation is that rare luxury vehicles are less represented in the dataset, making their prices more difficult to predict accurately.

**Overall, the model performs well across the dataset but is less reliable for extremely expensive and rare vehicles.** This is an important limitation to consider when using its predictions.

In [3]:
columns = ["fuel_category", "transmission", "body_type", "country_code"]

for column in columns:
    print(f"\n{column}:")
    print(sorted(df[column].dropna().unique()))


fuel_category:
['CNG', 'Diesel', 'Electric', 'Electric/Diesel', 'Electric/Gasoline', 'Ethanol', 'Gasoline', 'LPG', 'Others']

transmission:
['Automatic', 'Manual', 'Semi-automatic']

body_type:
['Compact', 'Convertible', 'Coupe', 'Off-Road/Pick-up', 'Other', 'Sedan', 'Station wagon', 'Transporter', 'Van']

country_code:
['AT', 'BE', 'DE', 'ES', 'FR', 'IT', 'LU', 'NL']


In [6]:
import json

# Get valid categories
columns = ["make", "fuel_category", "transmission", "body_type", "country_code"]

valid_categories = {}

for column in columns:
    valid_categories[column] = sorted(df[column].unique().tolist())

# Get valid models for each make
valid_categories["models_by_make"] = {}

for make, group in df.groupby("make"):
    valid_categories["models_by_make"][make] = sorted(group["model"].unique().tolist())

# Save categories to a JSON file
categories_path = project_root / "config" / "valid_categories.json"

with open(categories_path, "w", encoding="utf-8") as file:
    json.dump(valid_categories, file, indent=2, ensure_ascii=False)

print("Valid categories saved successfully!")

Valid categories saved successfully!
